# 01 · Data pipeline

Owner: data lead.

Runs stage 1 (clean + filter) and stage 2 (pairing) on the raw files, then the quality checks from the proposal.

In [ ]:
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, "..")
from config import *
from src import load, pairs as pairs_mod, models, evaluate

## Stage 1 — load, clean, filter, validate

In [ ]:
raw_files = load.find_raw_files(RAW_DIR)   # all years; load.find_raw_files(RAW_DIR, [2016]) for one
print(len(raw_files), "files")
df = load.load_chain_files(raw_files, load.FilterConfig(), out_dir=PROCESSED_DIR)
df.head()

In [ ]:
report = pd.read_csv(PROCESSED_DIR / "load_report.csv")
report

## Stage 2 — pair with next trading day, build target, assign splits

In [ ]:
pairs = pairs_mod.make_pairs_table(PROCESSED_DIR, PAIRS_PATH)
pairs.head()

## Quality checks

- filtered contracts per day (a dip flags a bad file; expect one in March 2020)
- Black-Scholes one-day hedging error distribution
- vendor delta vs. delta recomputed from the vendor's IV

In [ ]:
pairs.groupby("date").size().plot(title="filtered contracts per day"); plt.show()
pairs["err_bs"].describe()

In [ ]:
from src.bs import bs_delta
chk = pairs.sample(20000, random_state=0)
d_re = bs_delta(chk.S, chk.strike, chk["T"], 0.01, chk.iv, chk["type"].to_numpy())
plt.scatter(chk.delta, d_re, s=2, alpha=.3); plt.plot([-1,1],[-1,1],'r'); plt.xlabel("vendor delta"); plt.ylabel("recomputed"); plt.show()